In [30]:
# src/talaria/hermes_service/rag_processor/parser.py

import logging
from pathlib import Path
from typing import Optional, List
import re
from docx import Document
from pptx import Presentation
from PyPDF2 import PdfReader
import fitz

In [31]:
log = logging.getLogger(__name__)


def _parse_pdf(file_path: Path) -> Optional[str]:
    """ (⭐️ PyMuPDF) .pdf 파일에서 레이아웃을 보존하며 텍스트를 추출합니다. """
    try:
        text_parts: List[str] = []
        doc = fitz.open(file_path) # fitz.open() 사용
        
        if doc.is_encrypted:
            log.warning(f"암호화된 PDF는 건너뜁니다: {file_path.name}")
            return None
            
        # 모든 페이지 순회
        for page in doc:
            # "text" 옵션은 pypdf2보다 훨씬 더 나은 텍스트 순서를 반환합니다.
            text = page.get_text("text") 
            if text:
                text_parts.append(text)
        
        doc.close()
        
        raw_text = "\n\n".join(text_parts) # 페이지별로 두 줄 띄어쓰기

        # (⭐️) PyMuPDF로 추출한 텍스트도 동일한 정리 로직을 적용합니다.
        #      (이 로직은 이제 pypdf2보다 훨씬 깨끗한 텍스트에 적용됩니다)
        
        # 1. 3개 이상의 연속된 줄바꿈을 "진짜 문단 구분"인 \n\n 으로 통일
        cleaned_text = re.sub(r'\n{3,}', '\n\n', raw_text)
        
        # 2. (핵심) 문단 구분이 아닌 "단일" 줄바꿈(\n)만 공백(' ')으로 변경
        cleaned_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', cleaned_text)
        
        # 3. '공백/탭' 정리
        cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text).strip()
        
        return cleaned_text
        
    except Exception as e:
        log.error(f".pdf(PyMuPDF) 파일 파싱 실패 ({file_path.name}): {e}")
        return None


def _parse_docx(file_path: Path) -> Optional[str]:
    """ (⭐️) .docx (Word) 파일에서 '문단'과 '표' 텍스트를 모두 추출합니다. """
    try:
        text_parts: List[str] = []
        doc = Document(file_path)
        
        # --- 1. (기존) 문단(Paragraphs) 텍스트 추출 ---
        for para in doc.paragraphs:
            para_text = para.text.strip()
            if para_text:
                para_text = re.sub(r'[ \t]+', ' ', para_text)
                text_parts.append(para_text)
        
        # --- 2. (⭐️ 신규) 표(Tables) 텍스트 추출 ---
        for table in doc.tables:
            table_text_parts: List[str] = []
            for row in table.rows:
                row_text_parts: List[str] = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                        # 셀 안의 텍스트도 정리 (줄바꿈을 공백으로)
                        cell_text = re.sub(r'[\n\t]+', ' ', cell_text)
                        cell_text = re.sub(r' +', ' ', cell_text)
                        row_text_parts.append(cell_text)
                
                # 각 셀의 텍스트를 " | " (파이프)로 구분하여 한 줄로 만듭니다.
                if row_text_parts:
                    table_text_parts.append(" | ".join(row_text_parts))
            
            # (⭐️) 표(table) 하나가 끝나면, 그 내용을 하나의 블록으로 추가합니다.
            # (각 행은 \n 으로 구분됩니다)
            if table_text_parts:
                text_parts.append("\n".join(table_text_parts))

        # 3. (기존) 문단과 표 텍스트를 모두 \n\n (문단 구분)으로 합칩니다.
        return "\n\n".join(text_parts)
        
    except Exception as e:
        log.error(f".docx 파일 파싱 실패 ({file_path.name}): {e}")
        return None


# --------------------------------------------------
# (⭐️ 덮어쓰기) _parse_txt 함수 (이전 버전으로 복구)
# --------------------------------------------------
def _parse_txt(file_path: Path) -> Optional[str]:
    """ .txt 파일에서 텍스트를 추출하고 '문단'을 보존합니다. """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
    except UnicodeDecodeError:
        log.warning(f"UTF-8 디코딩 실패. '{file_path.name}' 파일을 latin-1로 재시도합니다.")
        try:
            with open(file_path, 'r', encoding='latin-1') as f:
                raw_text = f.read()
        except Exception as e:
            log.error(f".txt 파일 읽기 실패 ({file_path.name}): {e}")
            return None
    except Exception as e:
        log.error(f".txt 파일 읽기 실패 ({file_path.name}): {e}")
        return None

    # .txt의 문단 구분을 존중합니다.
    cleaned_text = re.sub(r'\n{3,}', '\n\n', raw_text)
    cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text).strip()
    return cleaned_text


# --- 메인 파서 함수 (이 함수는 수정할 필요가 없습니다) ---
def parse_document(file_path: str) -> Optional[str]:
    # ... (이전과 동일) ...
    try:
        path_obj = Path(file_path)
        file_ext = path_obj.suffix.lower()

        log.info(f"문서 파싱 시작: {path_obj.name}")

        if not path_obj.exists():
            log.error(f"파일을 찾을 수 없습니다: {file_path}")
            return None

        # 파일 확장자에 따라 적절한 파서 호출
        if file_ext == '.pdf':
            return _parse_pdf(path_obj)
        elif file_ext == '.docx':
            return _parse_docx(path_obj)
        # elif file_ext == '.pptx':
        #     return _parse_pptx(path_obj) # (pptx도 위 코드의 _parse_pptx로 교체해주세요)
        elif file_ext == '.txt':
            return _parse_txt(path_obj)
        else:
            log.warning(f"지원하지 않는 파일 형식입니다: {file_ext}")
            return None

    except Exception as e:
        log.error(f"'{file_path}' 문서 처리 중 예기치 않은 오류 발생: {e}", exc_info=True)
        return None

In [32]:
print("===== pdf test ======")
pdf_text = parse_document("./test_files/test.pdf")
print(pdf_text)
print("길이 : ", len(pdf_text))

print("\n===== docx test ======")
docx_text = parse_document("./test_files/test.docx")
print(docx_text)
print("길이 : ",len(docx_text))

print("\n===== txt test ======")
txt_text = parse_document("./test_files/test.txt")
print(txt_text)
print("길이 : ",len(txt_text))

===== pdf test ======
프로젝트 기획서: 문화 교류 프로젝트 1. 프로젝트 비전 및 컨셉 ●​ 프로젝트명: My Sarangbang Live (우리집 사랑방) ●​ 컨셉: 전 세계 친구들과 실시간으로 소통하는 온라인 '사랑방'입니다. 한국 고유의 '정(情)' 문화를 기반으로, 단순한 정보 전달을 넘어 사람과 사람 사이의 따뜻한 교감을 나누는 공간을 지향합니다. 참가자들은 평범한 한국인의 일상(요리, 취미, 동네 산책 등)에 직접 참여하며, 살아있는 문화를 체험하고 배우는 과정 속에서 자연스럽게 진정한 글로벌 친구가 됩니다. ●​ 핵심 가치: 이 프로젝트에서 기술은 목적이 아닌 사람을 연결하는 다리입니다. 저희는 완벽하고 유창한 소통보다, 언어의 장벽 앞에서 서로 돕고 웃을 수 있는 '서투름'의 가치를 존중합니다. 모든 기능은 참가자들이 활동에 몰입하고, 그 과정에서 자연스럽게 서로에게 가까워지는 풍부하고 감성적인 경험을 제공하는 데 초점을 맞춥니다. ●​ 타겟 사용자: ○​ 주요 타겟: K-Culture에 깊은 관심을 가진 팬, 즐겁게 한국어를 배우고 싶은 학습자, 관광 책자에서는 볼 수 없는 진짜 한국을 경험하고 싶은 글로벌 사용자. ○​ 확장 타겟: 지리적으로 떨어진 팀원들과 유대감을 쌓고 싶은 글로벌 기업, 몰입형 온라인 언어/문화 교육 프로그램을 도입하려는 교육 기관, 한국 문화에 대한 연결고리를 찾고 있는 해외 입양인 커뮤니티 등. 2. 핵심 기능 (Core Features) 1) '사전 교감'으로 더 깊어진 실시간 문화 체험 클래스 클래스 시작 전부터 호스트와 참가자 간의 유대감을 형성하고, 학습 효과와 몰입도를 극대화하는 기능입니다. ●​ 호스트의 '미리 나눔' 기능: 호스트는 '사랑방'을 개설할 때, 클래스 진행에 필요한 다양한 형태의 자료(PDF, 이미지, 텍스트로 작성된 요리 레시피, K-Pop 노래 가사, 수공예 도안 등)를 시스템에 미리 업로드할 수 있습니다. 이는 단순한 파일 공유를 넘어, 클래스에 대한 호스트의 정성과 